# B5.1 — Freeze `candidates_v2` (shared core) and the two final pipelines

**Module under test:** `src/pipeline/candidates_v2.py` (shared core) with `src/counting/pipeline_a.py` (GPV_R4+P[MixA], primary) and `src/counting/pipeline_b.py` (GPV_K-n+P[LM0], secondary).
Modes: `single` (one class: candidates_v1 + fallback ladder) and `multi` (several classes: U_both, B5.0 Part 4 decision).

Each check compares the module with reference counts **already saved on Drive** by earlier notebooks:

| Check | Reference | Expected | Runtime |
|---|---|---|---|
| 0 | per-class SAN functions (`run_san_vocab`, `run_san`, `run_san_d6`) | environment reproduces single-pass SAN exactly | GPU, ~2 min |
| A | B3PN per-image counts, train_dev400 (cached v1 inputs) | **exact**, 400/400, both systems | CPU-bound, ~20 min |
| B | B5.0 Part 4 one-class regression (v1 path, live, L4) | **exact** on 5 FSC train images; multi(1 class) == single | GPU, ~3 min |
| C | B5.0 Part 3b ladder V0 (FSC train empty-region images, L4) | primary **exact**; secondary reported (new) | GPU, ~3 min |
| D | B5.0 Part 4 `U_both` counts (OmniCount train, L4) | **exact**, both systems, 24 images | GPU, ~15 min |
| E | joint runner | standalone pipelines == joint, 2 images | GPU, ~2 min |

Live checks expect the **same GPU type as the references (L4)**; on another GPU small mask differences are possible and are reported.
Run order: Run all on an **L4** runtime.

In [1]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, installs (SAN + detectron2 + diffusers, protobuf last), repo
# ============================================================
import importlib, json, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import torch
assert torch.cuda.is_available(), "Use a GPU runtime (L4)."
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
B50 = RESULTS_ROOT / "B5_0_OmniCount_Region_Tests"
OUT_DIR = RESULTS_ROOT / "B5_1_Candidates_v2_Freeze"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OMNICOUNT_ROOT = PROJECT_ROOT / "datasets" / "OmniCount-191"
FSC147_IMAGE_DIR = PROJECT_ROOT / "datasets" / "FSC147" / "raw" / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| L4 expected for exact live checks")

DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
%cd /content
if not Path("/content/SAN").exists():
    !git clone -q https://github.com/MendelXu/SAN.git /content/SAN
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
!pip uninstall -y -q detectron2
!python -m pip install -q "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
for p in [str(REPO_DIR), "/content/SAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Repo:", REPO_COMMIT[:10])

GPU: NVIDIA L4 | L4 expected for exact live checks
/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 15.6 M

In [3]:
# ============================================================
# Step 0.3 — SAM2 (frozen commit), module imports, models
# ============================================================
import google.protobuf, importlib.metadata as md_meta
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart the session and Run all again."
import src.pipeline.b0 as b0
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()

import src.datasets as datasets_mod
import src.pipeline.semantic as semantic
import src.pipeline.prompts as prompts
import src.pipeline.candidates as C1
import src.pipeline.candidates_v2 as core
from src.counting import pipeline_a, pipeline_b

SAN_ROOT = Path("/content/SAN"); SAN_CKPT = SAN_ROOT / "output" / "san_vit_b_16.pth"
SAN_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not SAN_CKPT.exists():
    shutil.copy2(PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth", SAN_CKPT)
SAM2_CKPT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
M = core.load_models(SAN_ROOT, SAN_CKPT, SAM2_CKPT, systems=("A", "B"), device="cuda")
with open(OUT_DIR / "candidates_v2_frozen_config.json", "w") as f:
    json.dump({**core.frozen_config(), "repo_commit": REPO_COMMIT}, f, indent=2)
print(core.VERSION, "| models loaded:", "SAN, SAM2 predictor,", sorted(M.amg), "| normals:", M.normals is not None)
RESULTS = {}

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
/content


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 351M/351M [00:01<00:00, 257MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


model_index.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

candidates_v2 | models loaded: SAN, SAM2 predictor, ['LM0', 'MixA'] | normals: True


In [4]:
# ============================================================
# Check 0 — Environment: single-pass SAN == per-class SAN functions (1 FSC image)
# ============================================================
from PIL import Image
r0 = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv").iloc[0]
pil = Image.open(FSC147_IMAGE_DIR / r0["image_id"]).convert("RGB"); c = r0["category"]
U = core.union_vocabulary([c]); ev_vocab = semantic.expanded_vocabulary(c)
caps, shape = core.tiled_capture(M, pil, U)
new_tiled = core.tiled_subset(M, caps, shape, [U.index(x) for x in ev_vocab]).cpu().numpy()
w, h = pil.size; th, tw = int(np.ceil(core.TILE_FRAC * h)), int(np.ceil(core.TILE_FRAC * w))
ref = np.zeros((len(ev_vocab), h, w), np.float32); cnt = np.zeros((h, w), np.float32)
for y0 in (0, h - th):
    for x0 in (0, w - tw):
        ref[:, y0:y0 + th, x0:x0 + tw] += semantic.run_san_vocab(M.san, pil.crop((x0, y0, x0 + tw, y0 + th)), ev_vocab, "cuda")
        cnt[y0:y0 + th, x0:x0 + tw] += 1.0
ref /= cnt[None]
cls_u, sig_u, mp_low = core.san_capture(M, pil, U, low_res=True)
_, _, a3c_new, _, pts_new = core.class_evidence(cls_u, sig_u, mp_low, U, c)
t_ref, _ = semantic.run_san(M.san, pil, c, "cuda"); a3c_ref, _ = semantic.a3c_mask(t_ref)
mp_ref, tq_ref = semantic.run_san_d6(M.san, pil, c, "cuda")
pts_ref = np.asarray(prompts.dense_recovery_points(a3c_ref, mp_ref, tq_ref)[0], float).reshape(-1, 2)
chk0 = {"tiled max|diff|": float(np.abs(ref - new_tiled).max()),
        "tiled argmax agree": float((ref.argmax(0) == new_tiled.argmax(0)).mean()),
        "a3c agree": float((a3c_new == a3c_ref).mean()), "prompts": (len(pts_ref), len(pts_new)),
        "prompt max|dxy|": float(np.abs(np.sort(pts_ref, 0) - np.sort(pts_new, 0)).max()) if len(pts_ref) == len(pts_new) and len(pts_ref) else 0.0}
print(chk0)
RESULTS["check0"] = bool(chk0["tiled max|diff|"] < 1e-4 and chk0["tiled argmax agree"] > 0.9999 and chk0["a3c agree"] > 0.9999
                         and chk0["prompts"][0] == chk0["prompts"][1] and chk0["prompt max|dxy|"] < 0.5)
print("CHECK 0:", "PASS" if RESULTS["check0"] else "FAIL")
del caps, cls_u, sig_u, mp_low; torch.cuda.empty_cache()

{'tiled max|diff|': 0.0, 'tiled argmax agree': 1.0, 'a3c agree': 1.0, 'prompts': (22, 22), 'prompt max|dxy|': 0.0}
CHECK 0: PASS


In [5]:
# ============================================================
# Check A — Code-level, single-class: module counting layer on CACHED v1 inputs == B3PN (400 train images)
# ============================================================
from tqdm.auto import tqdm
td = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv").sort_values("dataset_index").reset_index(drop=True)
pn = pd.read_csv(RESULTS_ROOT / "B3PN_Part_and_Normals_Merge" / "B3PN_per_image.csv")
TD_PATHS = C1.CandidatePaths.standard(TD_ROOT)
rows = []
for r in tqdm(list(td.itertuples(index=False)), desc="Check A"):
    inp = C1.load_image_inputs(TD_PATHS, r.dataset_index, r.image_id)
    rows.append({"dataset_index": int(r.dataset_index),
                 "A": core.count_system("R4", *inp["sets"]["MixA"], inp["amax"], inp["d1"], inp["edge"]),
                 "B": core.count_system("Kn", *inp["sets"]["LM0"], inp["amax"], inp["d1"], inp["edge"])})
ca = pd.DataFrame(rows).merge(pn[["dataset_index", "GPV_R4+P[MixA]", "GPV_K-n+P[LM0]"]], on="dataset_index", validate="one_to_one")
eqA = (ca["A"] == ca["GPV_R4+P[MixA]"]); eqB = (ca["B"] == ca["GPV_K-n+P[LM0]"])
print(f"System A: {int(eqA.sum())}/{len(ca)} exact | System B: {int(eqB.sum())}/{len(ca)} exact")
if not (eqA.all() and eqB.all()):
    display(ca[~(eqA & eqB)].head(10))
RESULTS["checkA"] = bool(len(ca) == 400 and eqA.all() and eqB.all())
print("CHECK A:", "PASS" if RESULTS["checkA"] else "FAIL")
ca.to_csv(OUT_DIR / "B5_1_checkA_train_pool.csv", index=False)

Check A:   0%|          | 0/400 [00:00<?, ?it/s]

System A: 400/400 exact | System B: 400/400 exact
CHECK A: PASS


In [6]:
# ============================================================
# Check B — Live, single-class, ladder off: == B5.0 Part 4 one-class v1 path (5 FSC train images, L4)
#           + multi mode with one class == single mode
# ============================================================
ref4 = pd.read_csv(B50 / "part4_train_confirmation" / "B5_0_p4_regression_one_class.csv")
rows = []
for r in ref4.itertuples(index=False):
    img = FSC147_IMAGE_DIR / r.image_id
    s = core.count_image(M, img, [r.category], "single", ladder=False)["counts"]
    m_ = core.count_image(M, img, [r.category], "multi")["counts"]
    rows.append({"image_id": r.image_id, "category": r.category,
                 "A single": s["A"][r.category], "A multi(1)": m_["A"][r.category],
                 "B single": s["B"][r.category], "B multi(1)": m_["B"][r.category]})
cb = pd.DataFrame(rows)
cb["B ref (v1 path)"] = ref4["GPV_K-n+P[LM0] v1"].to_numpy()
cb["A ref (v1 path)"] = ref4["GPV_R4+P[MixA] v1"].to_numpy()
display(cb)
RESULTS["checkB"] = bool((cb["A single"] == cb["A ref (v1 path)"]).all() and (cb["B single"] == cb["B ref (v1 path)"]).all()
                         and (cb["A multi(1)"] == cb["A single"]).all() and (cb["B multi(1)"] == cb["B single"]).all())
print("CHECK B:", "PASS" if RESULTS["checkB"] else "FAIL", f"(GPU: {GPU_NAME})")
cb.to_csv(OUT_DIR / "B5_1_checkB_single_live.csv", index=False)

,image_id,category,A single,A multi(1),B single,B multi(1),B ref (v1 path),A ref (v1 path)
0,3263.jpg,matches,5,5,4,4,4,5
1,6379.jpg,beads,82,82,33,33,33,82
2,6834.jpg,roof tiles,17,17,18,18,18,17
3,4235.jpg,cupcake tray,28,28,24,24,24,28
4,5243.jpg,cupcakes,22,22,21,21,21,22


CHECK B: PASS (GPU: NVIDIA L4)


In [7]:
# ============================================================
# Check C — Ladder on the 5 FSC train empty-region images == B5.0 Part 3b V0 (primary)
#   Part 3b started the ladder at L1 (cached T4 mask sets were empty), so the ladder itself is tested
#   from L1 (R0 cleared); full single-class mode (from L0) is reported alongside.
# ============================================================
from PIL import Image
lad = pd.read_csv(B50 / "part3_end_to_end" / "B5_0_p3b_ladder_variants.csv")
lad = lad[lad["dataset"] == "FSC-147 train"].reset_index(drop=True)
rows = []
for r in lad.itertuples(index=False):
    path = FSC147_IMAGE_DIR / r.image_id; c = r.domain
    pil = Image.open(path).convert("RGB"); img_np = b0.load_image_rgb_cv2(path)
    ev = core.compute_evidence(M, pil, {c: c}); edge = core.normals_edge(M, pil)
    full = {s_: core.count_single(M, img_np, ev, c, s_, edge, ladder=True) for s_ in ("A", "B")}
    ev_l1 = {k: dict(v) for k, v in ev.items()}; ev_l1["R0"] = {c: np.zeros_like(ev["R0"][c])}
    from_l1 = {s_: core.count_single(M, img_np, ev_l1, c, s_, edge, ladder=True) for s_ in ("A", "B")}
    rows.append({"image_id": r.image_id, "category": c, "gt": r.gt,
                 "A ladder from L1": from_l1["A"][0], "A level": from_l1["A"][1], "A ref (3b V0)": r.V0, "ref level": r.V0_level,
                 "B ladder from L1": from_l1["B"][0], "B level": from_l1["B"][1],
                 "A full single": full["A"][0], "A full level": full["A"][1],
                 "B full single": full["B"][0], "B full level": full["B"][1]})
    torch.cuda.empty_cache()
cc = pd.DataFrame(rows); display(cc)
RESULTS["checkC"] = bool((cc["A ladder from L1"] == cc["A ref (3b V0)"]).all())
print("CHECK C:", "PASS" if RESULTS["checkC"] else "FAIL",
      "| 'full single' starts at L0: level 0 means the live L0 region produced masks (cached T4 set was empty)")
cc.to_csv(OUT_DIR / "B5_1_checkC_ladder.csv", index=False)

,image_id,category,gt,A ladder from L1,A level,A ref (3b V0),ref level,B ladder from L1,B level,A full single,A full level,B full single,B full level
0,2469.jpg,windows,286,321,1,321,L1,204,1,321,1,204,1
1,4130.jpg,mosaic tiles,365,149,1,149,L1,5,1,149,1,5,1
2,4133.jpg,mosaic tiles,150,151,2,151,L2,151,2,151,2,151,2
3,6974.jpg,bricks,366,283,1,283,L1,77,1,283,1,77,1
4,7675.jpg,windows,220,271,1,271,L1,216,1,271,1,216,1


CHECK C: PASS | 'full single' starts at L0: level 0 means the live L0 region produced masks (cached T4 set was empty)


In [8]:
# ============================================================
# Check D — Live, multi-label U_both == B5.0 Part 4 counts (24 OmniCount train images, 3 per domain, L4)
# ============================================================
train_ds = datasets_mod.OmniCount191Dataset(OMNICOUNT_ROOT, split="train", load_images=False)
VOCAB_TR = train_ds.domain_vocabulary
p4 = pd.read_csv(B50 / "part4_train_confirmation" / "B5_0_p4_subset.csv")
e4 = pd.read_csv(B50 / "part4_train_confirmation" / "B5_0_p4_counts.csv")
pick = p4.groupby("domain").head(3).reset_index(drop=True)
rows = []
for r in tqdm(list(pick.itertuples(index=False)), desc="Check D"):
    classes = VOCAB_TR[r.domain]
    prompt_of = {c: datasets_mod.omnicount_prompt_name(c) for c in classes}
    out = core.count_image(M, r.image_path, classes, "multi", prompt_of=prompt_of)["counts"]
    ref = e4[e4["image_id"] == r.image_id].set_index("class")
    for c in classes:
        rows.append({"image_id": r.image_id, "domain": r.domain, "class": c,
                     "A": out["A"][c], "A ref": int(ref.loc[c, "GPV_R4+P[MixA]|U_both"]),
                     "B": out["B"][c], "B ref": int(ref.loc[c, "GPV_K-n+P[LM0]|U_both"])})
cd = pd.DataFrame(rows)
eqA = (cd["A"] == cd["A ref"]); eqB = (cd["B"] == cd["B ref"])
print(f"Queries: {len(cd)} | A exact {int(eqA.sum())}/{len(cd)} | B exact {int(eqB.sum())}/{len(cd)}")
if not (eqA.all() and eqB.all()):
    display(cd[~(eqA & eqB)].head(15))
RESULTS["checkD"] = bool(eqA.all() and eqB.all())
print("CHECK D:", "PASS" if RESULTS["checkD"] else "FAIL", f"(GPU: {GPU_NAME})")
cd.to_csv(OUT_DIR / "B5_1_checkD_multi_live.csv", index=False)

Check D:   0%|          | 0/24 [00:00<?, ?it/s]

Queries: 399 | A exact 399/399 | B exact 399/399
CHECK D: PASS (GPU: NVIDIA L4)


In [9]:
# ============================================================
# Check E — Standalone pipelines (src.counting) == joint runner (2 OmniCount train images)
# ============================================================
rows = []
for r in pick.head(2).itertuples(index=False):
    classes = VOCAB_TR[r.domain]; prompt_of = {c: datasets_mod.omnicount_prompt_name(c) for c in classes}
    joint = core.count_image(M, r.image_path, classes, "multi", prompt_of=prompt_of)["counts"]
    a = pipeline_a.count_image(M, r.image_path, classes, "multi", prompt_of=prompt_of)["counts"]
    b = pipeline_b.count_image(M, r.image_path, classes, "multi", prompt_of=prompt_of)["counts"]
    rows.append({"image_id": r.image_id, "A == joint": a == joint["A"], "B == joint": b == joint["B"]})
ce = pd.DataFrame(rows); display(ce)
RESULTS["checkE"] = bool(ce["A == joint"].all() and ce["B == joint"].all())
print("CHECK E:", "PASS" if RESULTS["checkE"] else "FAIL")

,image_id,A == joint,B == joint
0,Birds/train/5,True,True
1,Birds/train/2,True,True


CHECK E: PASS


In [10]:
# ============================================================
# Summary — freeze record
# ============================================================
with open(OUT_DIR / "B5_1_freeze_record.json", "w") as f:
    json.dump({"module": core.VERSION, "repo_commit": REPO_COMMIT, "gpu": GPU_NAME,
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "checks": RESULTS, "all_pass": all(RESULTS.values()),
               "config_file": "candidates_v2_frozen_config.json"}, f, indent=2)
print(pd.Series(RESULTS).map({True: "PASS", False: "FAIL"}).to_string())
print("\nALL CHECKS:", "PASS — candidates_v2 frozen at commit " + REPO_COMMIT[:10] if all(RESULTS.values()) else "FAIL — send the failing tables")

check0    PASS
checkA    PASS
checkB    PASS
checkC    PASS
checkD    PASS
checkE    PASS

ALL CHECKS: PASS — candidates_v2 frozen at commit 2086087c5e
